# 07 · Modelos de sentimiento

**Objetivo:** entrenar y comparar los modelos de sentimiento del proyecto, del más sencillo al más complejo (M0 diccionario, M1 TF-IDF + red densa, M2 BERT propio sin preentrenar, M3a BERT propio preentrenado y M3b GPT propio preentrenado), sobre las frases etiquetadas de **Financial PhraseBank**. Todos usan la misma partición y las mismas métricas, para que la comparación sea justa.

**Entradas:**
- Financial PhraseBank (Malo et al., 2014), descargado en `data/phrasebank/`
- `data/cache/corpus_preentrenamiento.parquet` (notebook 06): texto sin etiquetas para preentrenar M3a y M3b

**Salidas:**
- `data/cache/phrasebank_particion.parquet`: las frases de 50Agree sin repetidas, con su parte (entrenamiento, validación o test) y si están en AllAgree
- *(el resto se irá añadiendo en cada paso)*

In [1]:
import os
import re
import zipfile

os.environ["KERAS_BACKEND"] = "torch"   # motor de Keras de este notebook (decisión del 20 oct 2026); fijarlo antes de importar keras

import keras
import pandas as pd
import requests
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import plotly.express as px
from dotenv import load_dotenv

from funciones import CARPETA_CACHE, SEMILLA

load_dotenv()
print("Keras", keras.__version__, "· motor:", keras.backend.backend())

Keras 3.15.1 · motor: torch


## 1. ¿Con qué frases etiquetadas aprenden los modelos? (Financial PhraseBank)

**Financial PhraseBank** (Malo, Sinha, Korhonen, Wallenius y Takala, 2014, *"Good debt or bad debt"*, [arXiv 1307.5336](https://arxiv.org/abs/1307.5336)) son ~4.800 frases de noticias financieras en inglés. Entre 5 y 8 anotadores con formación en finanzas etiquetaron cada una como **negativa, neutral o positiva**. Hay cuatro versiones según cuántos anotadores coincidieron: 50Agree (al menos la mitad), 66Agree, 75Agree y AllAgree (todos).

- **Fuente:** el zip original de los autores (`FinancialPhraseBank-v1.0.zip`), tal como está en la copia de Hugging Face `takala/financial_phrasebank`. Se descarga solo el zip, sin librerías intermedias.
- **Licencia:** CC BY-NC-SA 3.0. Uso académico sin fines comerciales, citando el artículo. `data/` no se sube a GitHub, así que el dataset no se redistribuye.

Se descarga una sola vez y se descomprimen solo los ficheros de los autores (el zip trae además carpetas ocultas de macOS).

In [2]:
URL_PHRASEBANK = "https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip"
CARPETA_PHRASEBANK = CARPETA_CACHE.parent / "phrasebank"
RUTA_ZIP = CARPETA_PHRASEBANK / "FinancialPhraseBank-v1.0.zip"
TAMANO_ZIP = 681_890   # bytes, comprobado el 21 oct 2026

if not RUTA_ZIP.exists():
    CARPETA_PHRASEBANK.mkdir(parents=True, exist_ok=True)
    respuesta = requests.get(URL_PHRASEBANK, timeout=60)
    respuesta.raise_for_status()
    RUTA_ZIP.write_bytes(respuesta.content)
if RUTA_ZIP.stat().st_size != TAMANO_ZIP:
    raise RuntimeError(f"El zip de PhraseBank no mide {TAMANO_ZIP} bytes: bórralo y vuelve a ejecutar")

with zipfile.ZipFile(RUTA_ZIP) as comprimido:
    ficheros = [nombre for nombre in comprimido.namelist() if nombre.startswith("FinancialPhraseBank-v1.0/") and nombre.endswith(".txt")]
    comprimido.extractall(CARPETA_PHRASEBANK, members=ficheros)
print("Ficheros:", [nombre.split("/")[1] for nombre in ficheros])

Ficheros: ['License.txt', 'README.txt', 'Sentences_50Agree.txt', 'Sentences_66Agree.txt', 'Sentences_75Agree.txt', 'Sentences_AllAgree.txt']


### 1.1 Cómo son los ficheros por dentro

Una frase por línea, con la etiqueta al final tras una `@`. La codificación es **latin-1**, no UTF-8: leído como UTF-8, el fichero falla en el byte 2.566. Así son las tres primeras líneas:

In [3]:
RUTA_50AGREE = CARPETA_PHRASEBANK / "FinancialPhraseBank-v1.0" / "Sentences_50Agree.txt"
for linea in RUTA_50AGREE.read_text(encoding="latin-1").splitlines()[:3]:
    print(linea)

According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .@neutral
Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies working in computer technologies and telecommunications , the statement said .@neutral
The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .@negative


Para leer cada variante se separa cada línea por la **última** `@` (por si alguna frase tuviera otra). Fíjate en que las frases traen **espacios antes de la puntuación** (*"Gran , the company"*): es como las dejaron los autores.

In [4]:
def leer_phrasebank(variante: str) -> pd.DataFrame:
    """Lee una variante de PhraseBank ('50Agree', 'AllAgree'…) y devuelve una tabla con frase y etiqueta."""
    ruta = CARPETA_PHRASEBANK / "FinancialPhraseBank-v1.0" / f"Sentences_{variante}.txt"
    filas = []
    for linea in ruta.read_text(encoding="latin-1").splitlines():
        frase, etiqueta = linea.rsplit("@", 1)
        filas.append({"frase": frase.strip(), "etiqueta": etiqueta.strip()})
    return pd.DataFrame(filas)


frases_50 = leer_phrasebank("50Agree")
frases_todas = leer_phrasebank("AllAgree")
print(f"50Agree: {len(frases_50):,} frases · AllAgree: {len(frases_todas):,} frases")

50Agree: 4,846 frases · AllAgree: 2,264 frases


### 1.2 ¿Cuántas frases hay de cada clase?

In [5]:
distribucion = pd.DataFrame({
    "50Agree": frases_50["etiqueta"].value_counts(),
    "50Agree (%)": (frases_50["etiqueta"].value_counts(normalize=True) * 100).round(1),
    "AllAgree": frases_todas["etiqueta"].value_counts(),
    "AllAgree (%)": (frases_todas["etiqueta"].value_counts(normalize=True) * 100).round(1),
})
distribucion

,50Agree,50Agree (%),AllAgree,AllAgree (%)
etiqueta,,,,
neutral,2879,59.4,1391,61.4
positive,1363,28.1,570,25.2
negative,604,12.5,303,13.4


### 1.3 ¿Cómo de largas son?

Longitud en palabras (contando como palabra cada trozo separado por espacios, signos de puntuación incluidos).

In [6]:
frases_50["palabras"] = frases_50["frase"].str.split().str.len()
print(frases_50.groupby("etiqueta")["palabras"].describe(percentiles=[0.5]).round(0)[["count", "min", "50%", "max"]])

grafico = px.histogram(frases_50, x="palabras", color="etiqueta", nbins=40, barmode="overlay",
                       title="Longitud de las frases de 50Agree, en palabras")
grafico.show(renderer="notebook_connected")

           count  min   50%   max
etiqueta                         
negative   604.0  5.0  22.0  56.0
neutral   2879.0  2.0  21.0  81.0
positive  1363.0  5.0  23.0  57.0


### 1.4 Ejemplos de cada clase

In [7]:
pd.set_option("display.max_colwidth", 160)
frases_50.groupby("etiqueta").sample(3, random_state=SEMILLA)[["etiqueta", "frase"]]

,etiqueta,frase
3790,negative,The company decided at the end of 2008 to temporarily shut down its ammonia plant in Billingham and extend the maintenance period at its Ince facility .
4670,negative,down to EUR5 .9 m H1 '09 3 August 2009 - Finnish media group Ilkka-Yhtyma Oyj ( HEL : ILK2S ) said today its net profit fell 45 % on the year to EUR5 .9 m i...
4797,negative,"The steelmaker said that the drop in profit was explained by the continuing economic uncertainty , mixed with the current drought in bank lending , resultin..."
1636,neutral,The mill is concentrating on getting the supercalendered line running satisfactorily before restarting its older newsprint line .
1117,neutral,Wellness services will be important in the future .
2775,neutral,"After completion of the acquisition , Poyry 's oil and gas expertise will cover multi-discipline engineering and consulting services across the complete oil..."
2310,positive,The fund at fair value will increase correspondingly .
2317,positive,This new partnership agreement represents a significant milestone for both parties .
1479,positive,`` The CHF is a great product .


### 1.5 ¿Hay frases repetidas?

Una frase repetida con **la misma** etiqueta solo pesa doble. Repetida con etiquetas **distintas** es peor: el modelo recibe dos respuestas contradictorias para la misma pregunta.

In [8]:
for nombre, tabla in [("50Agree", frases_50), ("AllAgree", frases_todas)]:
    repetidas = tabla[tabla.duplicated(subset="frase", keep=False)]
    etiquetas_por_frase = repetidas.groupby("frase")["etiqueta"].nunique()
    print(f"{nombre}: {repetidas['frase'].nunique()} frases repetidas ({len(repetidas)} filas)"
          f" · con etiquetas distintas: {(etiquetas_por_frase > 1).sum()}")

con_conflicto = frases_50.groupby("frase")["etiqueta"].nunique()
frases_50[frases_50["frase"].isin(con_conflicto[con_conflicto > 1].index)].sort_values("frase")

50Agree: 8 frases repetidas (16 filas) · con etiquetas distintas: 2
AllAgree: 5 frases repetidas (10 filas) · con etiquetas distintas: 0


,frase,etiqueta,palabras
78,TELECOMWORLDWIRE-7 April 2006-TJ Group Plc sells stake in Morning Digital Design Oy Finnish IT company TJ Group Plc said on Friday 7 April that it had signe...,neutral,42
79,TELECOMWORLDWIRE-7 April 2006-TJ Group Plc sells stake in Morning Digital Design Oy Finnish IT company TJ Group Plc said on Friday 7 April that it had signe...,positive,42
788,The Group 's business is balanced by its broad portfolio of sports and presence in all major markets .,positive,19
789,The Group 's business is balanced by its broad portfolio of sports and presence in all major markets .,neutral,19


### 1.6 ¿Aparece alguna frase de PhraseBank en nuestro corpus?

Si una frase del test estuviera también en el corpus de preentrenamiento del notebook 06, el BERT y el GPT la habrían "leído" antes de examinarse de ella. Se compara el texto **normalizado**: en minúsculas y **sin ningún espacio**, porque PhraseBank pone espacios antes de la puntuación (*"Gran , the"*) y el corpus no (*"Gran, the"*).

In [9]:
def normalizar(texto: str) -> str:
    """Texto en minúsculas y sin espacios, para comparar frases escritas con distinto espaciado."""
    return re.sub(r"\s+", "", texto.lower())


corpus = pd.read_parquet(CARPETA_CACHE / "corpus_preentrenamiento.parquet", columns=["texto", "fuente"])
corpus_normalizado = set(corpus["texto"].map(normalizar))
en_el_corpus = frases_50["frase"].map(normalizar).isin(corpus_normalizado)
print(f"Frases de PhraseBank (50Agree) que aparecen en el corpus: {en_el_corpus.sum()} de {len(frases_50):,}")

Frases de PhraseBank (50Agree) que aparecen en el corpus: 0 de 4,846


**Conclusión:**

- **4.846 frases en 50Agree y 2.264 en AllAgree**, muy desequilibradas: casi 6 de cada 10 son **neutrales** y solo 1 de cada 8 **negativa**. Un modelo que dijera siempre "neutral" acertaría el 59 % sin aprender nada; por eso la métrica principal será el F1 macro, que da el mismo peso a las tres clases.
- Son frases **cortas**: unas 22 palabras de mediana y como mucho 81. Para los Transformers bastará con un máximo de 128 tokens.
- **8 frases repetidas en 50Agree**, y **2 de ellas con etiquetas contradictorias** (la misma frase, una vez neutral y otra positiva). En AllAgree hay 5 repetidas, todas con la misma etiqueta.
- **Ninguna frase de PhraseBank aparece en nuestro corpus** de preentrenamiento: el preentrenamiento no habrá visto las frases del test.

## 2. ¿Con qué frases se entrena, se ajusta y se examina cada modelo? (partición)

### 2.1 Quitar las frases repetidas

Si una frase repetida cae una vez en entrenamiento y otra en test, el modelo se examina de algo que ya ha visto. Antes de partir:

- Las **2 frases con etiquetas contradictorias** se eliminan del todo: no sabemos cuál es la etiqueta buena.
- De las **demás repetidas** se deja una sola copia.

In [10]:
etiquetas_por_frase = frases_50.groupby("frase")["etiqueta"].nunique()
contradictorias = etiquetas_por_frase[etiquetas_por_frase > 1].index

frases = frases_50[~frases_50["frase"].isin(contradictorias)].drop_duplicates(subset="frase").reset_index(drop=True)
frases["en_allagree"] = frases["frase"].isin(frases_todas["frase"])
print(f"50Agree: {len(frases_50):,} filas → sin contradictorias ni repetidas: {len(frases):,} frases"
      f" ({len(frases_50) - len(frases)} filas fuera) · en AllAgree: {frases['en_allagree'].sum():,}")

50Agree: 4,846 filas → sin contradictorias ni repetidas: 4,836 frases (10 filas fuera) · en AllAgree: 2,259


### 2.2 Partición estratificada 60 / 20 / 20

Cada frase va a una de tres partes, al azar pero **manteniendo en cada parte la misma proporción de clases** (estratificada) y con la semilla fija:

| Parte | Para qué se usa |
|---|---|
| **Entrenamiento** (60 %) | El modelo aprende con estas frases |
| **Validación** (20 %) | Elegir hiperparámetros y decidir cuándo parar de entrenar |
| **Test** (20 %) | Examen final: **no se mira hasta la tabla final de todos los modelos (paso 9)** |

Los apuntes de clase usan el test como validación. Eso es una fuga de información: si se eligen las épocas mirando el test, el test deja de ser un examen limpio. Aquí no se hace.

Se parte en dos pasos: primero se separa el 20 % de test, y del 80 % restante se separa una cuarta parte (el 20 % del total) como validación.

In [11]:
resto, test = train_test_split(frases, test_size=0.2, stratify=frases["etiqueta"], random_state=SEMILLA)
entrenamiento, validacion = train_test_split(resto, test_size=0.25, stratify=resto["etiqueta"], random_state=SEMILLA)

particion = pd.concat([entrenamiento.assign(parte="entrenamiento"), validacion.assign(parte="validacion"),
                       test.assign(parte="test")], ignore_index=True)[["frase", "etiqueta", "parte", "en_allagree"]]
particion.to_parquet(CARPETA_CACHE / "phrasebank_particion.parquet", index=False)
print("Frases por parte:", particion["parte"].value_counts().to_dict())

Frases por parte: {'entrenamiento': 2901, 'test': 968, 'validacion': 967}


¿Tiene cada parte la misma proporción de clases? Y, para la medida de sensibilidad, ¿cuántas frases del test están también en AllAgree (todos los anotadores de acuerdo)?

In [12]:
proporciones = (pd.crosstab(particion["parte"], particion["etiqueta"], normalize="index") * 100).round(1)
proporciones["frases"] = particion["parte"].value_counts()
display(proporciones.loc[["entrenamiento", "validacion", "test"]])

test_allagree = particion[(particion["parte"] == "test") & particion["en_allagree"]]
print(f"Test: {(particion['parte'] == 'test').sum()} frases, de las que {len(test_allagree)} están en AllAgree."
      f" Se medirán aparte como sensibilidad: el mismo examen, solo con las frases en las que todos los anotadores coincidieron.")

etiqueta,negative,neutral,positive,frases
parte,,,,
entrenamiento,12.5,59.4,28.2,2901
validacion,12.5,59.4,28.1,967
test,12.5,59.4,28.1,968


Test: 968 frases, de las que 448 están en AllAgree. Se medirán aparte como sensibilidad: el mismo examen, solo con las frases en las que todos los anotadores coincidieron.


### 2.3 La referencia mínima: decir siempre "neutral"

Cualquier modelo tiene que superar a uno que no aprende nada y responde siempre la clase más frecuente. Se calcula sobre la **validación**, para no mirar el test hasta el final.

In [13]:
reales = particion.loc[particion["parte"] == "validacion", "etiqueta"]
siempre_neutral = ["neutral"] * len(reales)
print(f"Siempre neutral (validación) · exactitud: {accuracy_score(reales, siempre_neutral):.3f}"
      f" · F1 macro: {f1_score(reales, siempre_neutral, average='macro'):.3f}")

Siempre neutral (validación) · exactitud: 0.594 · F1 macro: 0.248


**La exactitud sale alta y el F1 macro muy bajo** porque la exactitud solo cuenta aciertos, y acertar todas las neutrales ya es el 59 %. El F1 macro hace la media de las tres clases con el mismo peso, y en negativa y positiva este "modelo" saca un 0.

**A tener en cuenta en M1 y M2 (sin decidir todavía):** con este desequilibrio, una red puede aprender a decir casi siempre "neutral" y conseguir buena exactitud. Una forma habitual de evitarlo son los **pesos de clase**: equivocarse en una negativa cuenta más que en una neutral. Se decidirá al entrenar M1, viendo los resultados en la validación.

**Conclusión:** quedan 4.836 frases repartidas 60/20/20 con la misma proporción de clases en cada parte, guardadas para que todos los modelos usen exactamente las mismas. El listón mínimo es un F1 macro de 0,25.

## Comprobaciones

In [14]:
# Lectura: las cifras de la ficha del dataset, tres etiquetas y ninguna frase vacía
assert len(frases_50) == 4846 and len(frases_todas) == 2264
for tabla in [frases_50, frases_todas]:
    assert set(tabla["etiqueta"]) == {"negative", "neutral", "positive"}
    assert (tabla["frase"].str.len() > 0).all()

# AllAgree es un subconjunto de 50Agree con las mismas etiquetas
pares_50 = set(zip(frases_50["frase"], frases_50["etiqueta"]))
assert all(par in pares_50 for par in zip(frases_todas["frase"], frases_todas["etiqueta"]))

# La normalización iguala frases que solo se diferencian en espacios y mayúsculas
assert normalizar("Gran , the company  GREW .") == normalizar("gran, the company grew.")

# Ninguna frase etiquetada está en el corpus de preentrenamiento
assert not en_el_corpus.any()
print("PhraseBank: OK")

PhraseBank: OK


In [15]:
# Partición: tamaños 60/20/20, sin frases repetidas dentro ni entre partes, y la misma proporción de clases
tamanos = particion["parte"].value_counts()
assert len(particion) == len(frases) == 4836
assert abs(tamanos["entrenamiento"] - 0.6 * len(particion)) <= 2
assert abs(tamanos["validacion"] - 0.2 * len(particion)) <= 2 and abs(tamanos["test"] - 0.2 * len(particion)) <= 2
assert not particion["frase"].map(normalizar).duplicated().any()
assert (proporciones[["negative", "neutral", "positive"]].max() - proporciones[["negative", "neutral", "positive"]].min()).max() < 1.0  # puntos %

# Las contradictorias no están, y en_allagree coincide con AllAgree
assert not particion["frase"].isin(contradictorias).any()
assert particion["en_allagree"].sum() == frases_todas["frase"].nunique()
print("Partición: OK")

Partición: OK
